# 随机森林分类：群众的智慧——多棵弱树组成强分类器
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/分类 | 源文件：随机森林_分类.py | 核心功能：Bagging + 随机特征选择、OOB 评估、特征重要性

## 概述

单棵决策树容易过拟合——它会记住训练数据中的每一个噪声。随机森林的解决方案非常直觉：**种一片森林，让所有树投票**。

随机森林在 Bagging（Bootstrap Aggregating）的基础上加入了**特征随机性**：每棵树不仅用不同的训练子集，每次分裂还只考虑随机选取的一部分特征。这两重随机性让各棵树尽可能"不同"，集成后的泛化能力远超单棵树。

脚本演示了随机森林的训练、OOB 评估、特征重要性分析，以及树数量对性能的影响。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

## 数学原理

### 1. Bagging 的方差缩减原理

**代码对应**：`RandomForestClassifier(n_estimators=100, max_features="sqrt")` 训练随机森林分类器。

随机森林基于 Bagging（Bootstrap Aggregating）思想。对 $B$ 棵树的预测取**多数投票**：

$$\hat{y} = \arg\max_{c} \sum_{b=1}^{B}\mathbb{I}(\hat{f}_b(\mathbf{x}) = c)$$

设单棵树的分类误差率为 $\epsilon$，树间相关系数为 $\rho$，$B$ 棵独立树的集成误差率为：

$$\epsilon_{\text{ensemble}} \approx \rho\epsilon + \frac{1-\rho}{B}\epsilon$$

当 $B \to \infty$ 时，误差率趋近 $\rho\epsilon$。因此**降低树间相关性 $\rho$** 比增加树的数量更重要。

### 2. 随机特征选择

**代码对应**：`max_features="sqrt"` 每次分裂只考虑 $\lfloor\sqrt{p}\rfloor$ 个随机特征。

两重随机性使各树多样化：

1. **样本随机**：每棵树用 Bootstrap 有放回抽样获得不同训练子集
2. **特征随机**：每次分裂只考虑随机选取的 $m$ 个特征（`max_features`）

| `max_features` | 树间相关性 $\rho$ | 集成效果 |
|----------------|------------------|---------|
| $p$（所有特征） | 高 | 较差 |
| $\sqrt{p}$（默认） | 中 | 最佳 |
| $1$ | 最低 | 较差（单棵树太弱） |

### 3. OOB 误差估计

**代码对应**：`oob_score=True` 启用 OOB 评分。

每棵树用 Bootstrap 采样训练，约 36.8% 的样本未被抽中（OOB）。样本 $i$ 的 OOB 预测为所有不包含 $i$ 的树的**多数投票**：

$$\hat{y}_i^{\text{OOB}} = \arg\max_c \sum_{b \in \mathcal{B}_i}\mathbb{I}(\hat{f}_b(\mathbf{x}_i) = c)$$

OOB 准确率是泛化准确率的无偏估计，等价于交叉验证但**完全免费**。

### 4. 特征重要性

**代码对应**：`rf.feature_importances_` 返回特征重要性。

**不纯度重要性**：特征 $j$ 的重要性为所有使用 $j$ 分裂的节点带来的基尼不纯度减少量的加权平均：

$$\text{Imp}(j) = \frac{1}{B}\sum_{b=1}^{B}\sum_{t \in T_b: \text{splits on } j}\frac{n_t}{N}\Delta\text{Gini}_t$$

### 5. 分类概率估计

随机森林的概率估计为各树预测概率的平均：

$$\hat{P}(y=c|\mathbf{x}) = \frac{1}{B}\sum_{b=1}^{B}\hat{P}_b(y=c|\mathbf{x})$$

其中 $\hat{P}_b(y=c|\mathbf{x})$ 为第 $b$ 棵树在 $\mathbf{x}$ 所在叶节点中类别 $c$ 的比例。

### 6. 与单棵决策树的对比

| 特性 | 单棵决策树 | 随机森林 |
|------|----------|---------|
| 训练准确率 | 1.0（无限制时） | < 1.0 |
| 测试准确率 | 较低（过拟合） | 较高（泛化好） |
| 方差 | 高 | 低（集成效应） |
| 偏差 | 低 | 略高（随机性降低单棵树质量） |
| 可解释性 | 高（可画树） | 低（黑盒集成） |

**代码对应**：代码中对比了单棵决策树和随机森林——单棵树训练 $R^2 = 1.0$ 但测试准确率较低，随机森林训练准确率更合理但测试准确率显著更高。

## 代码结构

| 段落 | 内容 |
|------|------|
| 数据 | 合成 3 分类数据（500 样本，15 特征） |
| 基础模型 | 100 棵树，OOB 评分 |
| 特征重要性 | 基于不纯度降低量排序 |
| n_estimators 对比 | 10 到 500 棵树的性能变化 |
| OOB 评估原理 | 每棵树约 63.2% 的 bootstrap 样本 + 36.8% 的袋外样本 |

### 1. 构造数据

运行 1. 构造数据 的代码，观察算法在该环节的行为。

In [ ]:
X, y = make_classification(
    n_samples=500, n_features=15, n_informative=8,
    n_classes=3, n_clusters_per_class=1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
# --- 赋值/配置 ---
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 2. 基础随机森林

运行 2. 基础随机森林 的代码，观察算法在该环节的行为。

In [ ]:
# n_estimators: 树的数量，越多越稳定但计算量越大
# max_features: 每棵树随机选择的特征数（分类默认 sqrt(n_features)）
rf = RandomForestClassifier(
    n_estimators=100, max_features="sqrt",
    oob_score=True, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

print("=== 随机森林分类 ===")
print(f"训练集准确率: {rf.score(X_train, y_train):.4f}")
print(f"测试集准确率: {rf.score(X_test, y_test):.4f}")
print(f"OOB（袋外）准确率: {rf.oob_score_:.4f}")
# OOB 评分：每棵树用未参与训练的样本评估，无需单独验证集

### 3. 特征重要性

分析各特征对模型预测的贡献度。

In [ ]:
print(f"\n=== 特征重要性（前 10）===")
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:10]
for rank, idx in enumerate(indices, 1):
    print(f"  {rank}. 特征{idx}: {importances[idx]:.4f}")

### 4. 不同参数对比

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== n_estimators 对比 ===")
for n in [10, 50, 100, 200, 500]:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    test_acc = rf_n.score(X_test, y_test)
# --- 继续 ---
    oob = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    oob.fit(X_train, y_train)
    print(f"  n={n:>4}: 测试准确率={test_acc:.4f}, OOB={oob.oob_score_:.4f}")

### 5. 袋外样本评估

在测试集上评估模型性能，关注关键指标。

In [ ]:
# oob_score=True 时，每棵树对未抽到的样本做预测，取投票结果
print("\n=== 袋外（OOB）评估原理 ===")
print(f"每棵树约使用 {int(0.632 * len(X_train))} 个样本训练（bootstrap 约 63.2%）")
print(f"剩余 ~{len(X_train) - int(0.632 * len(X_train))} 个样本作为该树的 OOB 样本")
print(f"OOB 评分等价于交叉验证，无需额外划分验证集")

### 6. 预测与分类报告

在分类任务上训练模型并评估性能。

In [ ]:
y_pred = rf.predict(X_test)
print(f"\n=== 分类报告 ===")
print(classification_report(y_test, y_pred))

### 7. 随机性来源

运行 7. 随机性来源 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 随机森林的两重随机性 ===")
print("1. 样本随机：每棵树通过 bootstrap 有放回抽样获得不同训练子集")
print("2. 特征随机：每次分裂只考虑随机选取的 max_features 个特征")
print("两重随机使各棵树多样化，降低过拟合风险")

## 关键代码解释

### OOB（袋外）评分——免费的交叉验证

```python
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1)
```

oob_score=True 是一个被严重低估的功能。每棵 bootstrap 树约使用 63.2% 的训练样本，剩余 36.8% 未被使用的样本叫"袋外样本"（Out-of-Bag）。对每个训练样本，用**未使用它的树**做预测并投票，得到的准确率就是 OOB 评分——它在数学上等价于交叉验证，但**不需要额外划分验证集**。

### 两重随机性

```python
rf = RandomForestClassifier(n_estimators=100, max_features="sqrt", ...)
```

1. **样本随机**：每棵树通过 bootstrap（有放回抽样）获得不同的训练子集
2. **特征随机**：每次分裂只考虑 max_features 个随机特征（分类默认 √n）

max_features 越小，树之间的差异越大（降低方差但增加偏差）。默认的 "sqrt" 在实践中效果很好。

### 特征重要性

```python
importances = rf.feature_importances_
```

基于每个特征在所有树中带来的不纯度降低总量。重要性高的特征对分类贡献大。

## 使用示例

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, oob_score=True, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
print(f"OOB 评分: {rf.oob_score_:.4f}")
print(dict(zip(feature_names, rf.feature_importances_)))
```

## 注意事项

1. **n_estimators 越多越好但收益递减**：100 棵通常够用，500 棵是保险选择
2. **OOB 评分只在 bootstrap=True 时可用**（默认即为 True）
3. **特征重要性对高基数特征有偏**：连续值特征比离散值特征更容易获得高重要性
4. **不需要特征缩放**：决策树不基于距离
5. **大数据集训练可能慢**：
_jobs=-1 并行化是必须的

## 总结与延伸

以上代码展示了 **随机森林 分类** 的完整流程。

**解读要点**：
- 关注 **准确率（Accuracy）** 和 **分类报告** 中的精确率/召回率/F1
- 如果训练准确率远高于测试准确率，说明存在过拟合
- 观察 **混淆矩阵**，找出模型最容易混淆的类别

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **ExtraTrees**（极端随机树）：分裂阈值也随机选取，比随机森林更快但方差更大
- **Isolation Forest**：基于随机森林思想的异常检测算法
- **特征重要性的替代品**：Permutation Importance 和 SHAP 值更可靠
- **与 XGBoost/LightGBM 对比**：随机森林不容易过拟合但精度通常略低，Boosting 系列精度更高但需要更小心调参
